# 🎨 GAN Studio — DCGAN vs WGAN-GP
## Tackling Mode Collapse in Generative Adversarial Networks
**Dataset**: Anime Faces 64×64 | **Platform**: Kaggle T4×2 GPU

| Section | Description |
|---------|-------------|
| 1 | Environment Setup & Config |
| 2 | Data Preparation & EDA |
| 3 | DCGAN Architecture |
| 4 | WGAN-GP Architecture |
| 5 | DCGAN Training Loop |
| 6 | WGAN-GP Training Loop |
| 7 | Visualisation & Loss Curves |
| 8 | Quantitative Evaluation (FID + IS) |
| 9 | Gradio App Deployment |

## Section 1 — Environment Setup & Imports

In [ ]:
# ============================================================
#  SECTION 1 — ENVIRONMENT SETUP & IMPORTS
# ============================================================

import os, math, random, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torchvision import datasets, transforms
import torchvision.utils as vutils

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()

# ── Device ────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print(f'[GPU] {n} GPU(s):')
        for i in range(n):
            p = torch.cuda.get_device_properties(i)
            print(f'  GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB')
        return torch.device('cuda:0')
    return torch.device('cpu')

DEVICE = get_device()

# ── Global config ─────────────────────────────────────────────
CFG = dict(
    DATA_ROOT   = '/kaggle/input/datasets/soumikrakshit/anime-faces',
    IMAGE_SIZE  = 64,
    NUM_WORKERS = 4,
    LATENT_DIM  = 100,
    CHANNELS    = 3,
    G_FEAT      = 64,
    D_FEAT      = 64,
    BATCH_SIZE  = 64,
    LR          = 2e-4,
    BETAS       = (0.5, 0.999),
    EPOCHS_DCGAN   = 50,
    EPOCHS_WGANGP  = 50,
    N_CRITIC    = 5,
    LAMBDA_GP   = 10,
    CKPT_DIR    = '/kaggle/working/checkpoints',
    SAMPLE_DIR  = '/kaggle/working/samples',
    CKPT_INTERVAL = 5,
    USE_AMP     = True,
)

os.makedirs(CFG['CKPT_DIR'], exist_ok=True)
os.makedirs(CFG['SAMPLE_DIR'], exist_ok=True)

def gpu_mem_info():
    if not torch.cuda.is_available(): return 'N/A'
    a = torch.cuda.memory_allocated()/(1024**3)
    r = torch.cuda.memory_reserved()/(1024**3)
    return f'Alloc:{a:.2f}GB Reserved:{r:.2f}GB'

print('[OK] Section 1 complete. Device:', DEVICE)

## Section 2 — Data Preparation

In [ ]:
# ============================================================
#  SECTION 2 — DATA PREPARATION
# ============================================================

import glob
from torch.utils.data import Dataset, DataLoader

DATA_ROOT  = CFG['DATA_ROOT']
IMAGE_SIZE = CFG['IMAGE_SIZE']

# ── Custom dataset ────────────────────────────────────────────
class AnimeFaceDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        exts = ('.jpg','.jpeg','.png','.webp')
        self.paths = [
            p for p in sorted(glob.glob(os.path.join(root,'**','*'), recursive=True))
            if os.path.isfile(p) and p.lower().endswith(exts)
        ]
        print(f'[Dataset] {len(self.paths):,} images loaded')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        try: img = Image.open(self.paths[idx]).convert('RGB')
        except: img = Image.new('RGB',(64,64),128)
        if self.transform: img = self.transform(img)
        return img, 0

# ── Transforms ────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE),
                       interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])

full_dataset = AnimeFaceDataset(DATA_ROOT, transform=transform_train)

train_loader = DataLoader(
    full_dataset,
    batch_size  = CFG['BATCH_SIZE'],
    shuffle     = True,
    num_workers = CFG['NUM_WORKERS'],
    pin_memory  = True,
    drop_last   = True,
)

# ── Fixed noise for visualisation ─────────────────────────────
FIXED_NOISE = torch.randn(64, CFG['LATENT_DIM'], 1, 1, device=DEVICE)

# ── EDA grid ─────────────────────────────────────────────────
batch, _ = next(iter(train_loader))
imgs = (batch[:64]*0.5+0.5).clamp(0,1)
grid = vutils.make_grid(imgs, nrow=8, padding=2)
plt.figure(figsize=(12,12))
plt.imshow(grid.permute(1,2,0).numpy())
plt.axis('off'); plt.title('Anime Faces — Training Batch', fontsize=14)
plt.savefig(os.path.join(CFG['SAMPLE_DIR'],'eda_batch.png'), dpi=100)
plt.show()

print(f'[OK] Section 2 complete. Batches/epoch: {len(train_loader)}')

## Section 3 — DCGAN Architecture

In [ ]:
# ============================================================
#  SECTION 3 — DCGAN ARCHITECTURE
# ============================================================

def weights_init_dcgan(m):
    cls = m.__class__.__name__
    if 'Conv' in cls:      nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cls:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

class DCGANGenerator(nn.Module):
    def __init__(self, latent_dim=100, feat_g=64, channels=3):
        super().__init__()
        self.net = nn.Sequential(
            # z(latent×1×1) → 4×4
            nn.ConvTranspose2d(latent_dim, feat_g*8, 4,1,0,bias=False),
            nn.BatchNorm2d(feat_g*8), nn.ReLU(True),
            # 4 → 8
            nn.ConvTranspose2d(feat_g*8, feat_g*4, 4,2,1,bias=False),
            nn.BatchNorm2d(feat_g*4), nn.ReLU(True),
            # 8 → 16
            nn.ConvTranspose2d(feat_g*4, feat_g*2, 4,2,1,bias=False),
            nn.BatchNorm2d(feat_g*2), nn.ReLU(True),
            # 16 → 32
            nn.ConvTranspose2d(feat_g*2, feat_g,   4,2,1,bias=False),
            nn.BatchNorm2d(feat_g), nn.ReLU(True),
            # 32 → 64
            nn.ConvTranspose2d(feat_g, channels,   4,2,1,bias=False),
            nn.Tanh(),
        )
    def forward(self, z): return self.net(z)

class DCGANDiscriminator(nn.Module):
    def __init__(self, feat_d=64, channels=3):
        super().__init__()
        self.net = nn.Sequential(
            # 64 → 32  (no BN)
            nn.Conv2d(channels, feat_d,   4,2,1,bias=False),
            nn.LeakyReLU(0.2, True),
            # 32 → 16
            nn.Conv2d(feat_d,   feat_d*2, 4,2,1,bias=False),
            nn.BatchNorm2d(feat_d*2), nn.LeakyReLU(0.2, True),
            # 16 → 8
            nn.Conv2d(feat_d*2, feat_d*4, 4,2,1,bias=False),
            nn.BatchNorm2d(feat_d*4), nn.LeakyReLU(0.2, True),
            # 8 → 4
            nn.Conv2d(feat_d*4, feat_d*8, 4,2,1,bias=False),
            nn.BatchNorm2d(feat_d*8), nn.LeakyReLU(0.2, True),
            # 4 → 1
            nn.Conv2d(feat_d*8, 1, 4,1,0,bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x): return self.net(x).view(-1)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

dcgan_G = DCGANGenerator(CFG['LATENT_DIM'], CFG['G_FEAT'], CFG['CHANNELS']).to(DEVICE)
dcgan_D = DCGANDiscriminator(CFG['D_FEAT'], CFG['CHANNELS']).to(DEVICE)
dcgan_G.apply(weights_init_dcgan)
dcgan_D.apply(weights_init_dcgan)

if torch.cuda.device_count() > 1:
    dcgan_G = nn.DataParallel(dcgan_G)
    dcgan_D = nn.DataParallel(dcgan_D)

print(f'DCGAN Generator     params: {count_parameters(dcgan_G):,}')
print(f'DCGAN Discriminator params: {count_parameters(dcgan_D):,}')
print('[OK] Section 3 complete.')

## Section 4 — WGAN-GP Architecture

In [ ]:
# ============================================================
#  SECTION 4 — WGAN-GP ARCHITECTURE
# ============================================================

import torch.autograd as autograd

class WGANGPGenerator(nn.Module):
    def __init__(self, latent_dim=100, feat_g=64, channels=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, feat_g*8, 4,1,0,bias=False),
            nn.BatchNorm2d(feat_g*8), nn.ReLU(True),
            nn.ConvTranspose2d(feat_g*8, feat_g*4, 4,2,1,bias=False),
            nn.BatchNorm2d(feat_g*4), nn.ReLU(True),
            nn.ConvTranspose2d(feat_g*4, feat_g*2, 4,2,1,bias=False),
            nn.BatchNorm2d(feat_g*2), nn.ReLU(True),
            nn.ConvTranspose2d(feat_g*2, feat_g,   4,2,1,bias=False),
            nn.BatchNorm2d(feat_g), nn.ReLU(True),
            nn.ConvTranspose2d(feat_g, channels,   4,2,1,bias=False),
            nn.Tanh(),
        )
    def forward(self, z): return self.net(z)

class WGANGPCritic(nn.Module):
    def __init__(self, feat_d=64, channels=3):
        super().__init__()
        def blk(ic, oc, bn=True):
            l = [nn.Conv2d(ic, oc, 4,2,1,bias=False)]
            if bn: l.append(nn.InstanceNorm2d(oc, affine=True))
            l.append(nn.LeakyReLU(0.2, True))
            return l
        self.net = nn.Sequential(
            *blk(channels,  feat_d,   bn=False),
            *blk(feat_d,    feat_d*2),
            *blk(feat_d*2,  feat_d*4),
            *blk(feat_d*4,  feat_d*8),
            nn.Conv2d(feat_d*8, 1, 4,1,0,bias=False),  # NO Sigmoid
        )
    def forward(self, x): return self.net(x).view(-1)

def gradient_penalty(critic, real, fake, device, lambda_gp=10.0):
    B   = real.size(0)
    eps = torch.rand(B,1,1,1,device=device).expand_as(real)
    interp = (eps*real + (1-eps)*fake).requires_grad_(True)
    scores = critic(interp)
    grads  = autograd.grad(
        scores, interp,
        grad_outputs=torch.ones_like(scores),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    grads = grads.view(B,-1)
    return lambda_gp * ((grads.norm(2,dim=1)-1)**2).mean()

wgan_G = WGANGPGenerator(CFG['LATENT_DIM'], CFG['G_FEAT'], CFG['CHANNELS']).to(DEVICE)
wgan_C = WGANGPCritic(CFG['D_FEAT'], CFG['CHANNELS']).to(DEVICE)

if torch.cuda.device_count() > 1:
    wgan_G = nn.DataParallel(wgan_G)
    wgan_C = nn.DataParallel(wgan_C)

print(f'WGAN-GP Generator params: {count_parameters(wgan_G):,}')
print(f'WGAN-GP Critic    params: {count_parameters(wgan_C):,}')
print('[OK] Section 4 complete.')

## Section 5 — DCGAN Training

In [ ]:
# ============================================================
#  SECTION 5 — DCGAN TRAINING LOOP
# ============================================================

criterion_dcgan = nn.BCELoss()
opt_G_dcgan = torch.optim.Adam(dcgan_G.parameters(), lr=CFG['LR'], betas=CFG['BETAS'])
opt_D_dcgan = torch.optim.Adam(dcgan_D.parameters(), lr=CFG['LR'], betas=CFG['BETAS'])
scaler_G_dcgan = GradScaler(enabled=CFG['USE_AMP'])
scaler_D_dcgan = GradScaler(enabled=CFG['USE_AMP'])

REAL_LABEL, FAKE_LABEL = 0.9, 0.0

dcgan_log = {'g_losses':[],'d_losses':[],'d_real_accs':[],'d_fake_accs':[],'epoch_times':[]}

def save_generated_grid(gen, noise, epoch, name, save_dir, nrow=8):
    gen.eval()
    with torch.no_grad():
        fake = gen(noise).detach().cpu()
    gen.train()
    imgs = (fake*0.5+0.5).clamp(0,1)
    grid = vutils.make_grid(imgs, nrow=nrow, padding=2)
    fig,ax = plt.subplots(figsize=(10,10))
    ax.imshow(grid.permute(1,2,0).numpy()); ax.axis('off')
    ax.set_title(f'{name} Epoch {epoch}',fontsize=11)
    plt.tight_layout()
    p = os.path.join(save_dir,f'{name}_epoch_{epoch:04d}.png')
    plt.savefig(p,dpi=90,bbox_inches='tight'); plt.close()

print('[DCGAN] Starting training ...')
for epoch in range(1, CFG['EPOCHS_DCGAN']+1):
    t0 = time.time()
    g_sum = d_sum = dx_sum = dgz_sum = nb = 0

    for real_imgs, _ in train_loader:
        real_imgs = real_imgs.to(DEVICE, non_blocking=True)
        B = real_imgs.size(0)
        rl = torch.full((B,), REAL_LABEL, dtype=torch.float, device=DEVICE)
        fl = torch.full((B,), FAKE_LABEL, dtype=torch.float, device=DEVICE)

        # --- D update ---
        dcgan_D.zero_grad()
        with autocast(enabled=CFG['USE_AMP']):
            out_r = dcgan_D(real_imgs)
            loss_r = criterion_dcgan(out_r, rl)
            z = torch.randn(B,CFG['LATENT_DIM'],1,1,device=DEVICE)
            fake = dcgan_G(z).detach()
            out_f = dcgan_D(fake)
            loss_f = criterion_dcgan(out_f, fl)
            loss_D = loss_r + loss_f
        scaler_D_dcgan.scale(loss_D).backward()
        scaler_D_dcgan.step(opt_D_dcgan)
        scaler_D_dcgan.update()

        # --- G update ---
        dcgan_G.zero_grad()
        with autocast(enabled=CFG['USE_AMP']):
            z = torch.randn(B,CFG['LATENT_DIM'],1,1,device=DEVICE)
            fake = dcgan_G(z)
            out_g = dcgan_D(fake)
            loss_G = criterion_dcgan(out_g, rl)
        scaler_G_dcgan.scale(loss_G).backward()
        scaler_G_dcgan.step(opt_G_dcgan)
        scaler_G_dcgan.update()

        g_sum += loss_G.item(); d_sum += loss_D.item()
        dx_sum += out_r.mean().item(); dgz_sum += out_g.mean().item()
        nb += 1

    el = time.time()-t0
    dcgan_log['g_losses'].append(g_sum/nb)
    dcgan_log['d_losses'].append(d_sum/nb)
    dcgan_log['d_real_accs'].append(dx_sum/nb)
    dcgan_log['d_fake_accs'].append(dgz_sum/nb)
    dcgan_log['epoch_times'].append(el)

    print(f'[DCGAN] Ep {epoch:3d}/{CFG["EPOCHS_DCGAN"]}  G:{g_sum/nb:.4f}  D:{d_sum/nb:.4f}  '
          f'D(x):{dx_sum/nb:.3f}  D(G):{dgz_sum/nb:.3f}  {el:.1f}s')

    save_generated_grid(dcgan_G, FIXED_NOISE, epoch, 'DCGAN', CFG['SAMPLE_DIR'])

    if epoch % CFG['CKPT_INTERVAL'] == 0 or epoch == CFG['EPOCHS_DCGAN']:
        torch.save({'epoch':epoch,'g_state':dcgan_G.state_dict(),
                    'd_state':dcgan_D.state_dict(),'log':dcgan_log},
                   os.path.join(CFG['CKPT_DIR'],f'dcgan_epoch_{epoch:04d}.pt'))

print('[OK] DCGAN Training complete.')

## Section 6 — WGAN-GP Training

In [ ]:
# ============================================================
#  SECTION 6 — WGAN-GP TRAINING LOOP
# ============================================================

opt_G_wgan = torch.optim.Adam(wgan_G.parameters(), lr=CFG['LR'], betas=CFG['BETAS'])
opt_C_wgan = torch.optim.Adam(wgan_C.parameters(), lr=CFG['LR'], betas=CFG['BETAS'])
scaler_G_wgan = GradScaler(enabled=CFG['USE_AMP'])
scaler_C_wgan = GradScaler(enabled=CFG['USE_AMP'])

wgan_log = {'g_losses':[],'c_losses':[],'gp_values':[],'epoch_times':[]}

print('[WGAN-GP] Starting training ...')
for epoch in range(1, CFG['EPOCHS_WGANGP']+1):
    t0 = time.time()
    g_sum = c_sum = gp_sum = nb = 0
    data_iter = iter(train_loader)

    while True:
        # ── Critic updates (N_CRITIC times) ──────────────────
        for _ in range(CFG['N_CRITIC']):
            try:    real_imgs,_ = next(data_iter)
            except: data_iter = iter(train_loader); real_imgs,_ = next(data_iter)
            real_imgs = real_imgs.to(DEVICE, non_blocking=True)
            B = real_imgs.size(0)
            wgan_C.zero_grad()
            with autocast(enabled=CFG['USE_AMP']):
                rs = wgan_C(real_imgs)
                z  = torch.randn(B,CFG['LATENT_DIM'],1,1,device=DEVICE)
                fake = wgan_G(z).detach()
                fs = wgan_C(fake)
                gp = gradient_penalty(wgan_C, real_imgs, fake, DEVICE, CFG['LAMBDA_GP'])
                loss_C = fs.mean() - rs.mean() + gp
            scaler_C_wgan.scale(loss_C).backward()
            scaler_C_wgan.step(opt_C_wgan)
            scaler_C_wgan.update()
            c_sum += loss_C.item(); gp_sum += gp.item()

        # ── Generator update ─────────────────────────────────
        try:    real_imgs,_ = next(data_iter)
        except: break
        B = real_imgs.size(0)
        wgan_G.zero_grad()
        with autocast(enabled=CFG['USE_AMP']):
            z    = torch.randn(B,CFG['LATENT_DIM'],1,1,device=DEVICE)
            fake = wgan_G(z)
            gs   = wgan_C(fake)
            loss_G = -gs.mean()
        scaler_G_wgan.scale(loss_G).backward()
        scaler_G_wgan.step(opt_G_wgan)
        scaler_G_wgan.update()
        g_sum += loss_G.item(); nb += 1

    d = max(nb,1)
    el = time.time()-t0
    wgan_log['g_losses'].append(g_sum/d)
    wgan_log['c_losses'].append(c_sum/(d*CFG['N_CRITIC']))
    wgan_log['gp_values'].append(gp_sum/(d*CFG['N_CRITIC']))
    wgan_log['epoch_times'].append(el)

    print(f'[WGAN-GP] Ep {epoch:3d}/{CFG["EPOCHS_WGANGP"]}  '
          f'G:{g_sum/d:.4f}  C:{c_sum/(d*CFG["N_CRITIC"]):.4f}  '
          f'GP:{gp_sum/(d*CFG["N_CRITIC"]):.4f}  {el:.1f}s')

    save_generated_grid(wgan_G, FIXED_NOISE, epoch, 'WGANGP', CFG['SAMPLE_DIR'])

    if epoch % CFG['CKPT_INTERVAL'] == 0 or epoch == CFG['EPOCHS_WGANGP']:
        torch.save({'epoch':epoch,'g_state':wgan_G.state_dict(),
                    'c_state':wgan_C.state_dict(),'log':wgan_log},
                   os.path.join(CFG['CKPT_DIR'],f'wgangp_epoch_{epoch:04d}.pt'))

print('[OK] WGAN-GP Training complete.')

## Section 7 — Visualisation & Loss Curves

In [ ]:
# ============================================================
#  SECTION 7 — VISUALISATION
# ============================================================

def smooth(v, w=5):
    if len(v)<w: return v
    k = np.ones(w)/w
    p = np.pad(v,(w//2,w//2),mode='edge')
    return np.convolve(p,k,mode='valid')[:len(v)]

# ── DCGAN loss curves ─────────────────────────────────────────
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('DCGAN Training Logs', fontsize=14, fontweight='bold')
ep = range(1,len(dcgan_log['g_losses'])+1)
axes[0].plot(ep, smooth(dcgan_log['g_losses']), color='#E74C3C', label='G Loss', lw=2)
axes[0].plot(ep, smooth(dcgan_log['d_losses']), color='#2980B9', label='D Loss', lw=2)
axes[0].set_title('G vs D Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, smooth(dcgan_log['d_real_accs']), color='#27AE60', label='D(x)', lw=2)
axes[1].plot(ep, smooth(dcgan_log['d_fake_accs']), color='#E67E22', label='D(G(z))', lw=2)
axes[1].axhline(0.5, color='grey', ls='--', alpha=0.6)
axes[1].set_title('D Output Scores'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG['SAMPLE_DIR'],'dcgan_losses.png'), dpi=130)
plt.show()

# ── WGAN-GP loss curves ───────────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle('WGAN-GP Training Logs', fontsize=14, fontweight='bold')
ep = range(1,len(wgan_log['g_losses'])+1)
axes[0].plot(ep, smooth(wgan_log['g_losses']),  color='#E74C3C', lw=2)
axes[0].set_title('Generator Loss'); axes[0].grid(alpha=0.3)
axes[1].plot(ep, smooth(wgan_log['c_losses']),  color='#2980B9', lw=2)
axes[1].set_title('Critic Loss (W-dist)'); axes[1].grid(alpha=0.3)
axes[2].plot(ep, smooth(wgan_log['gp_values']), color='#8E44AD', lw=2)
axes[2].set_title('Gradient Penalty'); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG['SAMPLE_DIR'],'wgangp_losses.png'), dpi=130)
plt.show()

# ── Side-by-side comparison ───────────────────────────────────
def gen_samples(model, n, lat, dev):
    model.eval()
    with torch.no_grad():
        z = torch.randn(n,lat,1,1,device=dev)
        imgs = model(z).cpu()
    model.train()
    return (imgs*0.5+0.5).clamp(0,1)

N = 10
d_imgs = gen_samples(dcgan_G, N, CFG['LATENT_DIM'], DEVICE)
w_imgs = gen_samples(wgan_G,  N, CFG['LATENT_DIM'], DEVICE)

fig = plt.figure(figsize=(20,5))
gs  = gridspec.GridSpec(2,N,hspace=0.1,wspace=0.05)
fig.suptitle('DCGAN (top) vs WGAN-GP (bottom)', fontsize=14, fontweight='bold')
for i in range(N):
    fig.add_subplot(gs[0,i]).imshow(d_imgs[i].permute(1,2,0).numpy()); plt.axis('off')
    fig.add_subplot(gs[1,i]).imshow(w_imgs[i].permute(1,2,0).numpy()); plt.axis('off')
plt.savefig(os.path.join(CFG['SAMPLE_DIR'],'comparison.png'), dpi=130, bbox_inches='tight')
plt.show()
print('[OK] Section 7 complete.')

## Section 8 — Quantitative Evaluation (FID + IS)

In [ ]:
# ============================================================
#  SECTION 8 — EVALUATION (FID + IS)
# ============================================================

# !pip install -q scipy
import scipy.linalg
from torchvision.models import inception_v3, Inception_V3_Weights

class InceptionFE(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.device = device
        m = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=False)
        m.fc = nn.Identity()
        self.m = m.eval().to(device)
        self.resize = transforms.Resize((299,299),
            interpolation=transforms.InterpolationMode.BILINEAR)

    @torch.no_grad()
    def forward(self, imgs):
        imgs = self.resize(imgs).to(self.device)*2.0-1.0
        f = self.m(imgs)
        return (f[0] if isinstance(f,tuple) else f).cpu()

def collect_feats(src, fe, n=5000, bs=64, lat=100, dev=None, is_gen=True):
    all_f=[]; cnt=0
    if is_gen:
        src.eval()
        while cnt<n:
            b = min(bs,n-cnt)
            z = torch.randn(b,lat,1,1,device=dev)
            with torch.no_grad(): imgs=(src(z).cpu()*0.5+0.5).clamp(0,1)
            all_f.append(fe(imgs).numpy()); cnt+=b
        src.train()
    else:
        for imgs,_ in src:
            imgs=(imgs*0.5+0.5).clamp(0,1)
            all_f.append(fe(imgs).numpy()); cnt+=imgs.size(0)
            if cnt>=n: break
    return np.concatenate(all_f)[:n]

def fid(fr, ff):
    ur,sr=fr.mean(0),np.cov(fr,rowvar=False)
    uf,sf=ff.mean(0),np.cov(ff,rowvar=False)
    d=(ur-uf).dot(ur-uf)
    cov,_=scipy.linalg.sqrtm(sr@sf,disp=False)
    if np.iscomplexobj(cov): cov=cov.real
    return float(d+np.trace(sr+sf-2*cov))

def inception_score(gen, n=5000, bs=64, lat=100, dev=None, splits=10):
    inc = inception_v3(weights=Inception_V3_Weights.DEFAULT,
                       transform_input=False).to(dev).eval()
    rsz = transforms.Resize((299,299))
    sf  = nn.Softmax(dim=1)
    preds=[]; cnt=0; gen.eval()
    with torch.no_grad():
        while cnt<n:
            b=min(bs,n-cnt)
            z=torch.randn(b,lat,1,1,device=dev)
            imgs=(gen(z).cpu()*0.5+0.5).clamp(0,1)
            imgs=rsz(imgs).to(dev)*2-1
            preds.append(sf(inc(imgs)).cpu().numpy()); cnt+=b
    gen.train()
    preds=np.concatenate(preds)[:n]
    scores=[]
    ss=n//splits
    for k in range(splits):
        p=preds[k*ss:(k+1)*ss]; py=p.mean(0)
        kl=(p*(np.log(p+1e-10)-np.log(py+1e-10))).sum(1).mean()
        scores.append(np.exp(kl))
    return float(np.mean(scores)),float(np.std(scores))

fe = InceptionFE(DEVICE)
N  = 5000
print('Collecting real features ...'); fr = collect_feats(train_loader,fe,N,is_gen=False)
print('DCGAN features ...');           fd = collect_feats(dcgan_G,fe,N,lat=CFG['LATENT_DIM'],dev=DEVICE)
print('WGAN-GP features ...');         fw = collect_feats(wgan_G, fe,N,lat=CFG['LATENT_DIM'],dev=DEVICE)

fid_d = fid(fr,fd); fid_w = fid(fr,fw)
print('Computing IS ...')
is_d_m,is_d_s = inception_score(dcgan_G,N,lat=CFG['LATENT_DIM'],dev=DEVICE)
is_w_m,is_w_s = inception_score(wgan_G, N,lat=CFG['LATENT_DIM'],dev=DEVICE)

print('\n'+'-'*44)
print(f'{"Metric":<18} {"DCGAN":>10} {"WGAN-GP":>12}')
print('-'*44)
print(f'{"FID ↓":<18} {fid_d:>10.2f} {fid_w:>12.2f}')
print(f'{"IS mean ↑":<18} {is_d_m:>10.3f} {is_w_m:>12.3f}')
print(f'{"IS std":<18} {is_d_s:>10.3f} {is_w_s:>12.3f}')
print('-'*44)
print('[OK] Section 8 complete.')

## Section 9 — Gradio App Deployment

In [ ]:
# ============================================================
#  SECTION 9 — GRADIO APP
# ============================================================
# !pip install -q gradio

import io, gradio as gr

def _to_pil(t):
    return Image.fromarray((t.permute(1,2,0).numpy()*255).astype(np.uint8))

def generate_images(model_name, n, seed):
    torch.manual_seed(int(seed))
    g = dcgan_G if model_name=='DCGAN' else wgan_G
    g.eval()
    with torch.no_grad():
        z=torch.randn(int(n),CFG['LATENT_DIM'],1,1,device=DEVICE)
        imgs=(g(z).cpu()*0.5+0.5).clamp(0,1)
    g.train()
    grid=vutils.make_grid(imgs,nrow=min(8,int(n)),padding=2)
    return _to_pil(grid), f'Model:{model_name}  N:{n}  Seed:{seed}'

def compare(n, seed):
    torch.manual_seed(int(seed))
    z=torch.randn(int(n),CFG['LATENT_DIM'],1,1,device=DEVICE)
    dcgan_G.eval(); wgan_G.eval()
    with torch.no_grad():
        d=(dcgan_G(z).cpu()*0.5+0.5).clamp(0,1)
        w=(wgan_G(z).cpu()*0.5+0.5).clamp(0,1)
    dcgan_G.train(); wgan_G.train()
    nr=min(int(n),8)
    dg=vutils.make_grid(d,nrow=nr,padding=2)
    wg=vutils.make_grid(w,nrow=nr,padding=2)
    comb=torch.cat([dg,wg],dim=1)
    return _to_pil(comb.clamp(0,1))

def loss_plot(name):
    log = dcgan_log if name=='DCGAN' else wgan_log
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    ep=range(1,len(log['g_losses'])+1)
    if name=='DCGAN':
        axes[0].plot(ep,log['g_losses'],label='G'); axes[0].plot(ep,log['d_losses'],label='D')
        axes[1].plot(ep,log['d_real_accs'],label='D(x)'); axes[1].plot(ep,log['d_fake_accs'],label='D(G(z))')
    else:
        axes[0].plot(ep,log['g_losses'],label='G'); axes[0].plot(ep,log['c_losses'],label='C')
        axes[1].plot(ep,log['gp_values'],label='GP')
    for ax in axes: ax.legend(); ax.grid(alpha=0.3)
    fig.suptitle(f'{name} Logs',fontsize=13)
    plt.tight_layout()
    buf=io.BytesIO(); plt.savefig(buf,format='png',dpi=100,bbox_inches='tight'); plt.close()
    buf.seek(0); return Image.open(buf)

with gr.Blocks(title='GAN Studio') as demo:
    gr.Markdown('# 🎨 GAN Studio — DCGAN vs WGAN-GP')
    with gr.Tab('Generate'):
        with gr.Row():
            m=gr.Dropdown(['DCGAN','WGAN-GP'],value='DCGAN',label='Model')
            n=gr.Slider(1,64,value=16,step=1,label='N Images')
            s=gr.Number(value=42,label='Seed',precision=0)
        btn=gr.Button('Generate',variant='primary')
        with gr.Row():
            out_img=gr.Image(type='pil',label='Output')
            out_txt=gr.Textbox(label='Info')
        btn.click(generate_images,[m,n,s],[out_img,out_txt])

    with gr.Tab('Compare'):
        with gr.Row():
            cn=gr.Slider(1,16,value=8,step=1,label='N')
            cs=gr.Number(value=0,label='Seed',precision=0)
        cbtn=gr.Button('Compare',variant='primary')
        cout=gr.Image(type='pil',label='DCGAN↑  WGAN-GP↓')
        cbtn.click(compare,[cn,cs],[cout])

    with gr.Tab('Loss Curves'):
        ldd=gr.Dropdown(['DCGAN','WGAN-GP'],value='DCGAN',label='Model')
        lbtn=gr.Button('Show',variant='primary')
        lout=gr.Image(type='pil')
        lbtn.click(loss_plot,[ldd],[lout])

    with gr.Tab('Metrics'):
        gr.Dataframe(
            headers=['Model','FID↓','IS↑','IS std'],
            value=[['DCGAN',  round(fid_d,2), round(is_d_m,3), round(is_d_s,3)],
                   ['WGAN-GP',round(fid_w,2), round(is_w_m,3), round(is_w_s,3)]],
        )

demo.launch(share=True, server_port=7860)
print('[OK] Gradio app running.')